# 🧠 CausalCrisis: Causal Multimodal Reasoning for Crisis Generalization

**Mô hình:** CausalCrisis — Mở rộng GEDA bằng Causal Inference (SCM + do-calculus)

**Notebook này chạy toàn bộ pipeline CausalCrisis:**
1. Setup Environment & Mount Drive
2. Download/Load Dataset CrisisMMD v2.0
3. Import CausalCrisis code
4. Sanity Check (forward pass + loss test)
5. Quick Test — 1 run duy nhất
6. Full Experiments — 60 runs (5 seeds x 3 tasks x 4 few-shot)
7. So sánh CausalCrisis vs GEDA baseline
8. Ablation Study — Đóng góp từng module
9. Qualitative Analysis (t-SNE, Domain Invariance)
10. Export kết quả về Drive

---

### Kiến trúc CausalCrisis (tóm tắt)
```
CLIP (frozen) → PCA → Projection → CausalDisentangler(C,S)
  → GRL + DomainClassifier(adversarial)
  → kNN Graph trên C features → GraphSAGE(2-layer)
  → SelfAttn → GCA → DiffAttn
  → CausalIntervention(do-calculus backdoor adjustment)
  → Multi-task Heads (Task 1,2,3)
  → CausalCrisisLoss (L_cls + L_adv + L_orth + L_recon + L_int)
```

### So sánh với các notebook khác
| Notebook | Mô hình | Ghi chú |
|:---------|:--------|:--------|
| `mm_class_experiment.ipynb` | Paper 1 — GNN Semi-supervised | Baseline GNN |
| `new_colab_notebook.ipynb` | Paper 2 — Differential Attention (GEDA) | Baseline DiffAttn |
| `geda_notebook.ipynb` | GEDA = GNN + DiffAttn kết hợp | Baseline kết hợp |
| **`causal_crisis_notebook.ipynb`** | **CausalCrisis = GEDA + Causal Inference** | **Notebook này** |

---
# 🛠️ PHASE 1: SETUP ENVIRONMENT

> ⚠️ **Chạy cell này ĐẦU TIÊN.** Khởi tạo tất cả imports, biến, và paths.

In [ ]:
#@title 1.1 Setup toàn bộ environment { display-mode: "form" }
#@markdown **Chọn nơi lưu kết quả:**
SOURCE_MODE = "download" #@param ["download", "drive"]
#@markdown - `download`: tải dataset bằng wget + lưu results trên Colab
#@markdown - `drive`: tải dataset bằng wget + mount Drive lưu results

import torch
import time
import os
import sys
import shutil
import numpy as np
from pathlib import Path

# === GPU Check ===
print("="*60)
print("  PHASE 1: Environment Setup")
print("="*60)
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {mem_gb:.1f} GB")
else:
    print("⚠️ No GPU! Runtime > Change runtime type > T4 GPU")

# === Paths (khởi tạo mặc định) ===
DATASET = None
DRIVE_DIR = None
LOCAL_RESULTS = "/content/causal_results"
RESULTS_CSV = f"{LOCAL_RESULTS}/all_results.csv"
DRIVE_RESULTS = LOCAL_RESULTS  # default
DRIVE_CKPTS = "/content/causal_checkpoints"

if SOURCE_MODE == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = "/content/drive/MyDrive/CrisisSummarization"
    DRIVE_RESULTS = f"{DRIVE_DIR}/causal_results"
    DRIVE_CKPTS = f"{DRIVE_DIR}/causal_checkpoints"
    RESULTS_CSV = f"{DRIVE_RESULTS}/all_results.csv"
    print(f"\n📂 Mode: Download (wget) + Save to Drive")
    print(f"   Drive dir: {DRIVE_DIR}")
else:
    print(f"\n📂 Mode: Download (wget) + Save địa phương")

# === Create directories ===
for d in [DRIVE_RESULTS, DRIVE_CKPTS, LOCAL_RESULTS]:
    os.makedirs(d, exist_ok=True)

print(f"\n  ✅ Setup complete! (mode={SOURCE_MODE})")

## 1.2 Cài đặt Dependencies

In [ ]:
# Cài dependencies — faiss-gpu tự detect CUDA version
import subprocess, sys
def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# Core packages
_install('open_clip_torch')
_install('scikit-learn')
_install('pandas')
_install('matplotlib')
_install('seaborn')

# FAISS — ưu tiên GPU cho hiệu năng tốt nhất
try:
    import torch
    cuda_ver = torch.version.cuda  # e.g. '12.4'
    if cuda_ver and cuda_ver.startswith('12'):
        _install('faiss-gpu-cu12')
        print(f'  ✅ faiss-gpu-cu12 (CUDA {cuda_ver})')
    elif cuda_ver and cuda_ver.startswith('11'):
        _install('faiss-gpu-cu11')
        print(f'  ✅ faiss-gpu-cu11 (CUDA {cuda_ver})')
    else:
        _install('faiss-cpu')
        print(f'  ⚠️ faiss-cpu (CUDA {cuda_ver} không hỗ trợ GPU build)')
except Exception as e:
    _install('faiss-cpu')
    print(f'  ⚠️ faiss-cpu (fallback: {e})')

import faiss
ngpus = faiss.get_num_gpus() if hasattr(faiss, 'get_num_gpus') else 0
print(f'  FAISS version: {faiss.__version__}, GPUs: {ngpus}')

---
# 📦 PHASE 2: DOWNLOAD DATASET

Tải **CrisisMMD v2.0** từ CrisisNLP bằng `wget` — **giống cách repo GNN gốc** ([jdnascim/mm-class-for-disaster-data-with-gnn](https://github.com/jdnascim/mm-class-for-disaster-data-with-gnn)).

> **Luôn dùng wget** để tải dataset, bất kể SOURCE_MODE là gì.

In [ ]:
import glob as _glob

# URL chính thức từ CrisisNLP (giống repo GNN)
DATASET_URL = 'https://crisisnlp.qcri.org/data/crisismmd/CrisisMMD_v2.0.tar.gz'
DATA_DIR = '/content/datasets'
TARGET_DIR = os.path.join(DATA_DIR, 'CrisisMMD_v2.0')
TAR_PATH = os.path.join(DATA_DIR, 'CrisisMMD_v2.0.tar.gz')

# === Bước 1: Tìm dataset đã có sẵn (cache) ===
search_paths = [
    TARGET_DIR,
    '/content/CrisisMMD_v2.0',
    '/content/data/CrisisMMD_v2.0',
]
if DRIVE_DIR:
    search_paths += [
        f'{DRIVE_DIR}/CrisisMMD_v2.0',
        '/content/drive/MyDrive/CrisisMMD_v2.0',
    ]

for p in search_paths:
    if os.path.isdir(p):
        DATASET = p
        break

# === Bước 2: Luôn dùng wget nếu chưa có ===
if DATASET is not None:
    print(f'  ✅ Dataset đã có (cache): {DATASET}')

else:
    # Download bằng wget (giống repo GNN) — ưu tiên hơn Drive
    os.makedirs(DATA_DIR, exist_ok=True)
    if not os.path.exists(TAR_PATH) or os.path.getsize(TAR_PATH) < 1.8e9:
        print(f'  📥 Downloading CrisisMMD v2.0 (~2GB)...')
        print(f'     URL: {DATASET_URL}')
        !wget -q --show-progress -c -O {TAR_PATH} {DATASET_URL}
    else:
        print(f'  ✅ Archive exists: {os.path.getsize(TAR_PATH)/1e9:.2f} GB')
    
    # Giải nén bằng system tar (nhanh hơn Python tarfile)
    print('  📦 Extracting...')
    !cd {DATA_DIR} && tar xzf CrisisMMD_v2.0.tar.gz
    DATASET = TARGET_DIR
    print(f'  ✅ Dataset ready: {DATASET}')

# === Bước 3: Verify ===
if DATASET and os.path.isdir(DATASET):
    assert os.path.isdir(DATASET), f'Dataset not found: {DATASET}'
    _n_img = len(_glob.glob(os.path.join(DATASET, '**', '*.jpg'), recursive=True))
    _n_img += len(_glob.glob(os.path.join(DATASET, '**', '*.png'), recursive=True))
    _tsv_dir = os.path.join(DATASET, 'crisismmd_datasplit_all')
    print(f'\n  📊 Dataset verified:')
    print(f'     Path:   {DATASET}')
    print(f'     Images: {_n_img:,}')
    if os.path.isdir(_tsv_dir):
        print(f'     TSVs:   {len(os.listdir(_tsv_dir))} files')
else:
    print(f'  ❌ Dataset chưa sẵn sàng!')

---
# 📥 PHASE 3: IMPORT CAUSALCRISIS CODE

**Tự động:** tìm trên Drive hoặc tải từ GitHub

In [ ]:
sys.path.insert(0, '/content')

GITHUB_REPO = 'https://raw.githubusercontent.com/Raghvendra-14/A-Multimodal-Approach-and-Dataset-to-Crisis-Summarization-in-Tweets/main'
required_files = ['src/models/causal_crisis_model.py', 'src/trainers/causal_crisis_trainer.py']
missing = []

for f in required_files:
    dst = f'/content/{f}'
    
    if os.path.exists(dst):
        print(f"  ✅ {f} — already in /content/")
        continue
    
    # Option 1: copy từ Drive
    if DRIVE_DIR:
        drive_src = f'{DRIVE_DIR}/{f}'
        if os.path.exists(drive_src):
            shutil.copy2(drive_src, dst)
            print(f"  ✅ {f} — copied from Drive")
            continue
    
    # Option 2: tải từ GitHub bằng wget
    missing.append(f)

if missing:
    print(f"\n  📥 Downloading missing files from GitHub...")
    for f in missing:
        url = f'{GITHUB_REPO}/{f}'
        dst = f'/content/{f}'
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        !wget -q -O {dst} {url}
        if os.path.exists(dst) and os.path.getsize(dst) > 100:
            print(f"  ✅ {f} — downloaded from GitHub")
        else:
            print(f"  ❌ {f} — download failed! Check URL: {url}")

# === Verify imports ===
try:
    from src.models.causal_crisis_model import (
        CausalCrisisModel, CausalCrisisLoss,
        CausalInterventionModule, CounterfactualRegularizer,
    )
    from src.trainers.causal_crisis_trainer import (
        CausalCrisisTrainer,
        extract_clip_features, apply_pca, build_knn_graph,
        run_causal_experiment, run_causal_all_experiments,
    )
    print("\n  ✅ All imports successful!")
except ImportError as e:
    print(f"\n  ❌ Import failed: {e}")
    print("     Kiểm tra causal_crisis_model.py và causal_crisis_trainer.py")

---
# 🧪 PHASE 4: SANITY CHECK — Forward Pass & Loss Test

## 4.1 Test Forward Pass (dữ liệu giả)

In [ ]:
import torch.nn.functional as F

B = 100  # batch size giả

model = CausalCrisisModel(
    img_dim=256, txt_dim=256, hidden_dim=512,
    causal_dim=256, spurious_dim=256,
    num_domains=7, dropout=0.3,
    use_graph=True, use_attention=True,
    use_mtl=True, use_causal=True,
    use_intervention=True,
).to(device)

img = torch.randn(B, 256).to(device)
txt = torch.randn(B, 256).to(device)
adj = torch.eye(B).to(device)
domains = torch.randint(0, 7, (B,)).to(device)

with torch.no_grad():
    outputs = model(img, txt, adj, domains, task="task1", grl_lambda=0.5)

print("=" * 60)
print("  SANITY CHECK: Forward Pass")
print("=" * 60)
for k, v in outputs.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:30s} shape={str(v.shape):20s} dtype={v.dtype}")
    elif isinstance(v, dict):
        for kk, vv in v.items():
            if isinstance(vv, torch.Tensor):
                print(f"  {k}.{kk:24s} shape={str(vv.shape):20s}")

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Parameters:")
print(f"  Total:     {total:>12,}")
print(f"  Trainable: {trainable:>12,}")
print(f"\n  ✅ Forward pass PASSED!")

## 4.2 Test Loss Function

In [ ]:
criterion = CausalCrisisLoss(
    alpha_adv=0.1, alpha_orth=0.01,
    alpha_recon=0.05, alpha_int=0.1,
    gamma=2.0, task_weights=(0.4, 0.3, 0.3),
).to(device)

targets = {
    "task1": torch.randint(0, 2, (B,)).to(device),
    "task2": torch.randint(0, 6, (B,)).to(device),
    "task3": torch.randint(0, 3, (B,)).to(device),
}
mask = torch.zeros(B, dtype=torch.bool).to(device)
mask[:50] = True

outputs = model(img, txt, adj, domains, task="task1", grl_lambda=0.5)
total_loss, loss_dict = criterion(outputs, targets, domains, mask)

print("=" * 60)
print("  SANITY CHECK: Loss Function")
print("=" * 60)
for k, v in loss_dict.items():
    val = v.item() if isinstance(v, torch.Tensor) else v
    print(f"  {k:20s}: {val:.6f}")

)))
if isinstance(total_loss, torch.Tensor):
    total_loss.backward()
    grad_norms = [(n, p.grad.norm().item()) for n, p in model.named_parameters()
                  if p.grad is not None]
    print(f"\n  Gradients computed for {len(grad_norms)} parameters")
    print(f"  Max grad norm: {max(g for _, g in grad_norms):.6f}")
    print(f"  Min grad norm: {min(g for _, g in grad_norms):.6f}")

print(f"\n  ✅ Loss computation PASSED!")

del model, criterion, outputs, loss_dict
torch.cuda.empty_cache()

---
# 🔬 PHASE 5: QUICK TEST — 1 Run Thật

## 5.1 Extract CLIP Features + Domain Labels

In [ ]:
print("="*60)
print("  PHASE 5: Feature Extraction")
print("="*60)

for task in ["task1"]:
    for split in ["train", "test"]:
        print(f"\n--- {task}/{split} ---")
        img_f, txt_f, labels, events = extract_clip_features_with_domain(
            DATASET, task=task, split=split, device=device
        )
        print(f"  img: {img_f.shape}, txt: {txt_f.shape}")
        print(f"  labels: {len(set(labels))} unique")
        print(f"  events: {len(set(events))} unique — {sorted(set(events))}")

## 5.2 Chạy 1 Experiment (task1, N=250, seed=42)

In [ ]:
print("="*60)
print("  PHASE 5: Quick Test — 1 Run")
print("="*60)

t_start = time.time()

result = run_causal_experiment(
    dataset_path=DATASET,
    task="task1",
    n_labeled=250,
    seed=42,
    device=device,
    max_epochs=100,
    patience=30,
    results_csv=f"{LOCAL_RESULTS}/quick_test.csv",
    use_graph=True,
    use_attention=True,
    use_mtl=True,
    use_causal=True,
    use_intervention=True,
    variant_name="causal_quicktest",
    use_causal_graph=True,
)

elapsed = time.time() - t_start

print(f"\n{'='*60}")
print(f"  QUICK TEST RESULT")
print(f"{'='*60}")
print(f"  Weighted F1: {result['weighted_f1']:.4f}")
print(f"  Macro F1:    {result['macro_f1']:.4f}")
print(f"  Best epoch:  {result.get('best_epoch', 'N/A')}")
print(f"  Time:        {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"{'='*60}")

if result['weighted_f1'] > 0.5:
    print("  ✅ Model converges! Ready for full experiments.")
else:
    print("  ⚠️ F1 thấp — kiểm tra lại hyperparameters hoặc data.")

---
# 🚀 PHASE 6: FULL EXPERIMENTS — 60 Runs

**Cấu hình:** 5 seeds × 3 tasks × 4 few-shot sizes = 60 runs  
**Ước tính thời gian:** ~4-8 giờ trên Colab T4/L4  
**Kết quả lưu tự động** (Drive hoặc local)

In [ ]:
print("="*60)
print("  PHASE 6: Full CausalCrisis Experiments")
print("="*60)

t_start = time.time()

run_causal_all_experiments(
    dataset_path=DATASET,
    seeds=(42, 123, 456, 789, 1024),
    tasks=("task1", "task2", "task3"),
    few_shot_sizes=(50, 100, 250, 500),
    device=device,
    results_csv=RESULTS_CSV,
    variant_name="causal_full",
    use_causal=True,
    use_intervention=True,
    use_causal_graph=True,
)

elapsed = time.time() - t_start
print(f"\n  ⏱️ Total time: {elapsed/3600:.1f} hours")

if os.path.exists(RESULTS_CSV):
    backup = RESULTS_CSV.replace('.csv', f'_backup_{int(time.time())}.csv')
    shutil.copy2(RESULTS_CSV, backup)
    print(f"  💾 Backup saved: {backup}")

---
# 📊 PHASE 7: SO SÁNH CAUSALCRISIS vs GEDA

In [ ]:
import pandas as pd
from scipy import stats

print("="*70)
print("  PHASE 7: CausalCrisis vs GEDA Comparison")
print("="*70)

# === Load GEDA results ===
geda_csv_candidates = ["/content/geda_results/all_results.csv"]
if DRIVE_DIR:
    geda_csv_candidates.insert(0, f"{DRIVE_DIR}/geda_results/all_results.csv")

geda_csv = None
for c in geda_csv_candidates:
    if os.path.exists(c):
        geda_csv = c
        break

dfs = {}
if geda_csv:
    dfs["geda_full"] = pd.read_csv(geda_csv)
    print(f"  ✅ GEDA: {len(dfs['geda_full'])} runs from {geda_csv}")
else:
    print(f"  ⚠️ GEDA results NOT FOUND — chạy geda_notebook.ipynb trước")

if os.path.exists(RESULTS_CSV):
    dfs["causal_full"] = pd.read_csv(RESULTS_CSV)
    print(f"  ✅ CausalCrisis: {len(dfs['causal_full'])} runs loaded")
else:
    print(f"  ⚠️ CausalCrisis results NOT FOUND — chạy Phase 6 trước")

In [ ]:
# === Comparison Table ===
task_names = {"task1": "Informativeness", "task2": "Humanitarian", "task3": "Damage"}

if len(dfs) >= 2:
    df_all = pd.concat(dfs.values(), ignore_index=True)

    print("\n" + "="*70)
    print("  COMPARISON TABLE: Weighted F1 (mean ± std)")
    print("="*70)

    for task in ["task1", "task2", "task3"]:
        print(f"\n--- {task_names[task]} ---")
        print(f"{'Model':35s} {'N=50':>10s} {'N=100':>10s} {'N=250':>10s} {'N=500':>10s}")
        print("-" * 70)

        for model_name in ["geda_full", "causal_full"]:
            label = "GEDA" if model_name == "geda_full" else "CausalCrisis"
            row = f"{label:35s}"
            for n in [50, 100, 250, 500]:
                mask = (
                    (df_all['model'] == model_name) &
                    (df_all['task'] == task) &
                    (df_all['few_shot'] == n)
                )
                vals = df_all.loc[mask, 'weighted_f1'].values * 100
                if len(vals) > 0:
                    row += f"  {vals.mean():.1f}±{vals.std():.1f}"
                else:
                    row += f"       N/A"
            print(row)
elif "causal_full" in dfs:
    df = dfs["causal_full"]
    print("\n  CausalCrisis Results (standalone):")
    print(df.groupby(['task', 'few_shot'])['weighted_f1'].agg(['mean', 'std']).round(4))
else:
    print("  ⚠️ Chưa có results. Chạy Phase 6 trước!")

In [ ]:
# === Statistical Significance Tests ===
if len(dfs) >= 2:
    print("\n" + "="*70)
    print("  SIGNIFICANCE TESTS: CausalCrisis vs GEDA (t-test, α=0.05)")
    print("="*70)

    for task in ["task1", "task2", "task3"]:
        print(f"\n--- {task_names[task]} ---")
        for n in [50, 100, 250, 500]:
            geda_vals = df_all.loc[
                (df_all['model'] == 'geda_full') & (df_all['task'] == task) &
                (df_all['few_shot'] == n), 'weighted_f1'
            ].values
            causal_vals = df_all.loc[
                (df_all['model'] == 'causal_full') & (df_all['task'] == task) &
                (df_all['few_shot'] == n), 'weighted_f1'
            ].values

            if len(geda_vals) >= 2 and len(causal_vals) >= 2:
                t_stat, p_val = stats.ttest_ind(causal_vals, geda_vals)
                diff = (causal_vals.mean() - geda_vals.mean()) * 100
                pooled_std = np.sqrt((geda_vals.std()**2 + causal_vals.std()**2) / 2)
                cohens_d = (causal_vals.mean() - geda_vals.mean()) / pooled_std if pooled_std > 0 else 0
                sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
                print(f"  N={n:3d}: Δ={diff:+.1f}%  p={p_val:.4f} {sig}  d={cohens_d:.2f}")
            else:
                print(f"  N={n:3d}: insufficient data")
else:
    print("  (Skipped — cần cả GEDA và CausalCrisis results)")

---
# 🔍 PHASE 8: ABLATION STUDY

4 biến thể CausalCrisis:
1. `no_causal` — Tắt Disentangler (= GEDA gốc)
2. `no_intervention` — Có Disentangler nhưng không do-calculus
3. `no_causal_graph` — Graph trên raw features (không causal)
4. `causal_full` — Full CausalCrisis ⭐ (đã chạy Phase 6)

In [ ]:
print("="*60)
print("  PHASE 8: Ablation Study")
print("="*60)

ABLATION_CSV = f"{DRIVE_RESULTS}/ablation_results.csv"

ablation_configs = {
    "no_causal": {
        "use_causal": False, "use_intervention": False, "use_causal_graph": False,
    },
    "no_intervention": {
        "use_causal": True, "use_intervention": False, "use_causal_graph": True,
    },
    "no_causal_graph": {
        "use_causal": True, "use_intervention": True, "use_causal_graph": False,
    },
}

for variant, config in ablation_configs.items():
    print(f"\n{'─'*40}")
    print(f"  Variant: {variant}")
    print(f"{'─'*40}")
    
    run_causal_all_experiments(
        dataset_path=DATASET,
        seeds=(42, 123, 456, 789, 1024),
        tasks=("task1", "task2", "task3"),
        few_shot_sizes=(250,),
        device=device,
        results_csv=ABLATION_CSV,
        variant_name=variant,
        **config,
    )

print(f"\n  ✅ Ablation complete! Results: {ABLATION_CSV}")

In [ ]:
# === Ablation Results Table ===
if os.path.exists(ABLATION_CSV):
    df_abl = pd.read_csv(ABLATION_CSV)
    if os.path.exists(RESULTS_CSV):
        df_full = pd.read_csv(RESULTS_CSV)
        df_full_250 = df_full[df_full['few_shot'] == 250]
        df_abl = pd.concat([df_abl, df_full_250], ignore_index=True)
    
    print("\n" + "="*70)
    print("  ABLATION TABLE: Weighted F1 @ N=250 (mean ± std)")
    print("="*70)
    
    variant_order = ["no_causal", "no_causal_graph", "no_intervention", "causal_full"]
    variant_labels = {
        "no_causal": "A1: No Disentangle (=GEDA)",
        "no_causal_graph": "A2: Raw Graph (no causal)",
        "no_intervention": "A3: No do-calculus",
        "causal_full": "A4: Full CausalCrisis ⭐",
    }
    
    print(f"\n{'Variant':40s} {'Task1':>10s} {'Task2':>10s} {'Task3':>10s}")
    print("-" * 75)
    for v in variant_order:
        label = variant_labels.get(v, v)
        row = f"{label:40s}"
        for task in ["task1", "task2", "task3"]:
            mask = (df_abl['model'] == v) & (df_abl['task'] == task)
            vals = df_abl.loc[mask, 'weighted_f1'].values * 100
            if len(vals) > 0:
                row += f"  {vals.mean():.1f}±{vals.std():.1f}"
            else:
                row += f"       N/A"
        print(row)
else:
    print("  ⚠️ Chạy Phase 8 ablation trước!")

---
# 🎨 PHASE 9: QUALITATIVE ANALYSIS

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

print("="*60)
print("  PHASE 9: Qualitative Analysis")
print("="*60)

best_model = CausalCrisisModel(
    img_dim=256, txt_dim=256, hidden_dim=512,
    causal_dim=256, spurious_dim=256,
    num_domains=7, dropout=0.3,
    use_graph=True, use_attention=True,
    use_mtl=True, use_causal=True,
    use_intervention=True,
).to(device)

ckpt_path = f"{DRIVE_CKPTS}/best_causal_model.pt"
if os.path.exists(ckpt_path):
    best_model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f"  Loaded checkpoint: {ckpt_path}")
else:
    print(f"  ⚠️ No checkpoint — using random weights for demo")

img_f, txt_f, labels, events = extract_clip_features_with_domain(
    DATASET, task="task1", split="test", device=device
)

pca = PCA(n_components=256)
img_pca = pca.fit_transform(img_f)
txt_pca = pca.fit_transform(txt_f)

best_model.eval()
with torch.no_grad():
    adj = torch.eye(len(img_pca)).to(device)
    domains_t = torch.zeros(len(img_pca), dtype=torch.long).to(device)
    unique_events = sorted(set(events))
    event_to_idx = {e: i for i, e in enumerate(unique_events)}
    for i, e in enumerate(events):
        domains_t[i] = event_to_idx[e]
    
    outputs = best_model(
        torch.FloatTensor(img_pca).to(device),
        torch.FloatTensor(txt_pca).to(device),
        adj, domains_t, task="task1", grl_lambda=1.0
    )

print(f"  Outputs ready for {len(img_pca)} samples")

In [ ]:
# === t-SNE Visualization ===
if 'causal' in outputs and isinstance(outputs['causal'], dict):
    causal_data = outputs['causal']
    fig, axes = plt.subplots(2, 2, figsize=(18, 16))
    fig.suptitle('CausalCrisis: Causal vs Spurious Features', fontsize=16, fontweight='bold')
    
    plots = [
        ('C_v', causal_data.get('C_v'), 'Causal Image', labels),
        ('S_v', causal_data.get('S_v'), 'Spurious Image', events),
        ('C_t', causal_data.get('C_t'), 'Causal Text', labels),
        ('S_t', causal_data.get('S_t'), 'Spurious Text', events),
    ]
    
    for idx, (key, feat, title, color_labels) in enumerate(plots):
        ax = axes[idx // 2][idx % 2]
        if feat is not None:
            feat_np = feat.cpu().numpy()
            tsne = TSNE(n_components=2, random_state=42, perplexity=30)
            emb = tsne.fit_transform(feat_np)
            unique_labels = sorted(set(color_labels))
            colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
            for i, l in enumerate(unique_labels):
                mask = np.array([cl == l for cl in color_labels])
                ax.scatter(emb[mask, 0], emb[mask, 1], c=[colors[i]], label=l[:20], alpha=0.6, s=15)
            ax.set_title(title, fontsize=12)
            ax.legend(fontsize=7, loc='best', ncol=2)
        else:
            ax.text(0.5, 0.5, f'{key} not available', ha='center', va='center', transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
    
    plt.tight_layout()
    save_path = f"{DRIVE_RESULTS}/tsne_causal_vs_spurious.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  💾 Saved: {save_path}")
else:
    print("  ⚠️ Causal features not in outputs")

In [ ]:
# === Domain Invariance Check ===
if 'domain_logits_causal' in outputs:
    dom_logits = outputs['domain_logits_causal']
    dom_preds = dom_logits.argmax(dim=-1).cpu().numpy()
    dom_true = domains_t.cpu().numpy()
    dom_acc = (dom_preds == dom_true).mean()
    chance = 1.0 / len(unique_events)
    
    print("="*60)
    print("  DOMAIN INVARIANCE CHECK")
    print("="*60)
    print(f"  Domain accuracy on C features: {dom_acc*100:.1f}%")
    print(f"  Chance level:                  {chance*100:.1f}%")
    if dom_acc < chance + 0.1:
        print(f"  ✅ Causal features are DOMAIN-INVARIANT")
    else:
        print(f"  ⚠️ Causal features leak domain info — tăng alpha_adv")
else:
    print("  ⚠️ domain_logits_causal not in outputs")

---
# 💾 PHASE 10: EXPORT & SUMMARY

In [ ]:
print("="*70)
print("  PHASE 10: Final Summary")
print("="*70)

if os.path.exists(RESULTS_CSV):
    df = pd.read_csv(RESULTS_CSV)
    print(f"\n  📊 CausalCrisis: {len(df)} runs")
    task_names = {"task1": "Informativeness", "task2": "Humanitarian", "task3": "Damage"}
    
    print(f"\n{'Task':25s} {'N':>5s} {'wF1 (mean±std)':>16s} {'macF1':>16s}")
    print("-" * 65)
    for task in ["task1", "task2", "task3"]:
        for n in [50, 100, 250, 500]:
            mask = (df['task'] == task) & (df['few_shot'] == n)
            wf1 = df.loc[mask, 'weighted_f1'].values * 100
            mf1 = df.loc[mask, 'macro_f1'].values * 100 if 'macro_f1' in df.columns else [0]
            if len(wf1) > 0:
                print(f"  {task_names[task]:23s} {n:5d}  {wf1.mean():.1f} ± {wf1.std():.1f}     "
                      f"{np.mean(mf1):.1f} ± {np.std(mf1):.1f}")

# Nếu mode=download, tải results về máy
if SOURCE_MODE == "download":
    from google.colab import files
    for f in [RESULTS_CSV]:
        if os.path.exists(f):
            files.download(f)
            print(f"  📥 Downloaded: {f}")

print(f"\n📁 Results saved at: {RESULTS_CSV}")
print(f"\n{'='*70}")
print(f"  🎉 CausalCrisis experiments complete!")
print(f"{'='*70}")